[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/moeabench/blob/main/examples/full_analysis.ipynb)

# MoeaBench Masterclass: Comprehensive Algorithm Analysis
----------------------------------------------------------

This notebook demonstrates the full capabilities of `moeabench` for algorithmic diagnostics, focusing on the comparison between **NSGA-II** and **NSGA-III** on a multi-objective problem (**DTLZ2**).

We will cover:
1. **Experimentation**: Launching and managing multi-run experiments.
2. **Physical Metrics (`mb.metrics`)**: Raw indicators like IGD+, GD+, and Hypervolume.
3. **Statistical Analysis (`mb.stats`)**: Aggregating results and calculating probabilities.
4. **Visual Diagnostics (`mb.view`)**: Topology shapes, convergence history, and performance contrast.
5. **Stratification (`mb.stats.strata`)**: Taxonomy of solution castes and Earth Mover's Distance.
6. **Scientific Audit (`mb.diagnostics`)**: The FAIR biopsy and clinical Q-Scores.

In [ ]:
# Install moeabench from GitHub (Required for Colab)
try:
    from moeabench import mb
except ImportError:
    !pip install --quiet git+https://github.com/monacofj/moeabench.git
    from moeabench import mb

In [ ]:
from moeabench import mb
import numpy as np

print(f"MoeaBench version: {mb.system.version()}")

## 1. Setup & Experimentation

We use `DTLZ2` with $M=3$ objectives. 
- **NSGA-II** (Crowding Distance) is highly effective in 3D but begins to show diversity gaps compared to reference-point approaches.
- **NSGA-III** (Reference Points) provides a more uniform structural scaffold.

In [ ]:
# Problem Definition (M=3 matches library clinical baselines)
mop = mb.mops.DTLZ2(M=3)

# Experiment 1: NSGA-II
exp2 = mb.experiment()
exp2.name = "NSGA-II"
exp2.mop = mop
exp2.moea = mb.moeas.NSGA2(population=92, generations=200)

# Experiment 2: NSGA-III
exp3 = mb.experiment()
exp3.name = "NSGA-III"
exp3.mop = mop
exp3.moea = mb.moeas.NSGA3(population=92, generations=200)

# Execute Multi-run (3 runs each for stability)
print("Running NSGA-II...")
exp2.run(repeat=3)

print("Running NSGA-III...")
exp3.run(repeat=3)

## 2. Physical Metrics & Statistics

We analyze the raw results using standard Pareto-compliant indicators.

In [ ]:
# IGD+ Analysis
igdp2 = mb.metrics.igdplus(exp2)
igdp3 = mb.metrics.igdplus(exp3)

print("NSGA-II IGD+ Summary:")
igdp2.report()

print("\nNSGA-III IGD+ Summary:")
igdp3.report()

# Performance Probability (Statistical dominance of NSGA-III over NSGA-II)
prob = mb.stats.perf_probability(igdp3, igdp2)
print(f"\nProbability NSGA-III > NSGA-II (based on IGD+): {float(prob):.2%}")

## 3. Visual Analysis

Visualization is key for understanding objective spaces.

In [ ]:
# Topology Shape (Visual Geometry)
print("Visualizing Objective Space (Final Pareto Fronts)...")
mb.view.topo_shape(exp2.front(), exp3.front(), 
                   title="NSGA-II vs NSGA-III (3D DTLZ2 Fronts)")

# Convergence History (HV Progress)
hv2 = mb.metrics.hv(exp2)
hv3 = mb.metrics.hv(exp3)
mb.view.perf_history(hv2, hv3, title="Convergence Stability (Hypervolume History)")

# Performance Contrast (Boxplot distribution)
# Shows the statistical spread of closeness to the optimal manifold.
mb.view.perf_spread(exp2, exp3, title="Quality Distribution (Closeness Contrast)")

## 4. Stratification Analysis

We investigate the internal hierarchy of the final populations.

In [ ]:
# Stratification into Caste/Tiers
strata3 = mb.stats.strata(exp3)
print("NSGA-III Stratification (Distribution):")
for r, freq in strata3.ranks.items():
    print(f" - Rank {r}: {freq*100:.1f}% solutions")

# Caste Visualization
mb.view.strat_caste(exp3.front(), title="NSGA-III Caste Distribution")

# EMD (Earth Mover's Distance) - Structural distance between algorithm profiles
dist = mb.stats.emd(mb.stats.strata(exp2), mb.stats.strata(exp3))
print(f"\nGlobal Topographic Displacement (EMD): {float(dist):.4f}")

## 5. Clinical Audit (FAIR Diagnostics)

The final step is the scientific audit that interprets the results through the FAIR (Finite Approximation-Induced Resolution) framework.

In [ ]:
# Execute Clinical Audit
audit2 = mb.diagnostics.audit(exp2)
audit3 = mb.diagnostics.audit(exp3)

print("--- NSGA-II BIOPSY REPORT ---")
audit2.report()

print("\n--- NSGA-III BIOPSY REPORT ---")
audit3.report()

## 6. Synthesis

In this 3D scenario, observe how:
- **NSGA-III** typically shows high **Q_BALANCE** and **Q_COVERAGE** due to its reference points providing a uniform structural scaffold.
- **NSGA-II** often achieves excellent **Q_CLOSENESS**, but may show subtle diversity gaps in higher dimensions.

This diagnostic profile allows you to choose the algorithm based on the specific **clinical failure modes** your problem can tolerate.